In [2]:
# 4. GrayBox_Attack.ipynb
# Goal:
# 1) Train a surrogate model on GTSRB
# 2) Generate adversarial examples using the surrogate model
# 3) Evaluate transferability on the target ResNet-18 model

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import copy
import random
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from tqdm import tqdm

from sklearn.model_selection import train_test_split

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

경로설정

In [5]:
base_dir = '/content/drive/MyDrive/26sp_ML/PR1'
zip_path = os.path.join(base_dir, 'GTSRB.zip')
extract_dir = '/content/GTSRB'

target_model_path = os.path.join(base_dir, 'resnet18_gtsrb_best_full_model.pth')
save_dir = os.path.join(base_dir, 'graybox_results')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

In [6]:
# unzip if needed
if not os.path.exists(os.path.join(extract_dir, 'Train.csv')):
    print('Extracting GTSRB.zip...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)

print('extract_dir:', extract_dir)
print('target_model_path exists:', os.path.exists(target_model_path))

Extracting GTSRB.zip...
extract_dir: /content/GTSRB
target_model_path exists: True


csv load/ split


In [7]:
train_csv_path = os.path.join(extract_dir, 'Train.csv')
test_csv_path  = os.path.join(extract_dir, 'Test.csv')

train_df_full = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

train_df, val_df = train_test_split(
    train_df_full,
    test_size=0.1,
    random_state=42,
    stratify=train_df_full['ClassId']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

num_classes = train_df_full['ClassId'].nunique()

print('Train:', train_df.shape)
print('Val  :', val_df.shape)
print('Test :', test_df.shape)
print('Num classes:', num_classes)

Train: (35288, 8)
Val  : (3921, 8)
Test : (12630, 8)
Num classes: 43


Dataset class

In [8]:
class GTSRBDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, return_path=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.return_path = return_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['Path'])
        label = int(row['ClassId'])

        image = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        if self.return_path:
            return image, label, row['Path']
        return image, label

transform

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

pixel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataloader

In [10]:
batch_size = 64
num_workers = 2
pin_memory = torch.cuda.is_available()

train_dataset = GTSRBDataset(train_df, extract_dir, transform=train_transform)
val_dataset   = GTSRBDataset(val_df, extract_dir, transform=eval_transform)
test_dataset  = GTSRBDataset(test_df, extract_dir, transform=eval_transform)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory
)

# attack generation용 loader
attack_dataset = GTSRBDataset(test_df, extract_dir, transform=pixel_transform, return_path=True)
attack_loader = DataLoader(
    attack_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory
)

device / normalize helper

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def normalize_batch(x):
    return (x - mean) / std

device: cuda


Target model load (baseline ResNet-18)

In [12]:
target_model = torch.load(target_model_path, map_location=device, weights_only=False)
target_model = target_model.to(device)
target_model.eval()

print('Loaded target model from:', target_model_path)

Loaded target model from: /content/drive/MyDrive/26sp_ML/PR1/resnet18_gtsrb_best_full_model.pth


surrogate model

In [13]:
#1. simple CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 224
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [14]:
surrogate_model = SimpleCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(surrogate_model.parameters(), lr=1e-3, weight_decay=1e-4)

surrogate 학습 함수

In [15]:
def run_one_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    if optimizer is None:
        model.eval()
    else:
        model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for batch in loader:
        images, labels = batch[:2]
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if optimizer is not None:
            optimizer.zero_grad()

        with torch.set_grad_enabled(optimizer is not None):
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            if optimizer is not None:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    return epoch_loss, epoch_acc

surrogate 학습

In [ ]:
num_epochs = 10
best_val_acc = 0.0
best_surrogate_wts = copy.deepcopy(surrogate_model.state_dict())

surrogate_model_path = os.path.join(save_dir, 'surrogate_cnn_best.pth')

for epoch in range(num_epochs):
    train_loss, train_acc = run_one_epoch(
        surrogate_model, train_loader, criterion, optimizer=optimizer, device=device
    )
    val_loss, val_acc = run_one_epoch(
        surrogate_model, val_loader, criterion, optimizer=None, device=device
    )

    print(f'[Epoch {epoch+1:02d}/{num_epochs}] '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_surrogate_wts = copy.deepcopy(surrogate_model.state_dict())
        torch.save(best_surrogate_wts, surrogate_model_path)
        print('Best surrogate model saved.')

In [ ]:
surrogate_model.load_state_dict(best_surrogate_wts)
surrogate_model.eval() #evaluation mode
print('Best surrogate val acc:', best_val_acc) #validation set quality check

surrogate clean test accuracy check

최종 surrogate 성능 (test set 이용)

In [ ]:
test_loss, surrogate_test_acc = run_one_epoch(
    surrogate_model, test_loader, criterion, optimizer=None, device=device
)

print(f'Surrogate clean test accuracy: {surrogate_test_acc:.4f}')

target clean test accuracy check

공격 전 성능 (baseline) test set 이용

In [ ]:
def evaluate_model(model, loader, device='cpu'):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            images, labels = batch[:2]
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total, correct, total

In [ ]:
target_clean_acc, correct, total = evaluate_model(target_model, test_loader, device=device)
print(f'Target clean test accuracy: {target_clean_acc:.4f} ({correct}/{total})')

Gray-box FGSM 생성 + target 평가
- gradient : surrogate_model 에서 계산
- evaluation : target_model 에서 계산

In [ ]:
eps_list_255 = [2, 4, 8, 16]
eps_list = [e / 255.0 for e in eps_list_255]

graybox_fgsm_results = []

surrogate_model.eval()
target_model.eval()

for e255, eps in zip(eps_list_255, eps_list):
    total = 0
    target_adv_correct = 0
    target_clean_correct = 0
    attack_success_on_clean = 0

    for images, labels, paths in tqdm(attack_loader, desc=f'Gray-box FGSM eps={e255}/255'):
        images = images.to(device, non_blocking=True)   # pixel space [0,1]
        labels = labels.to(device, non_blocking=True)

        # gradient는 surrogate model 기준으로
        images_for_attack = images.clone().detach().requires_grad_(True)

        surrogate_model.zero_grad(set_to_none=True)
        outputs_sur = surrogate_model(normalize_batch(images_for_attack))
        loss = criterion(outputs_sur, labels)
        loss.backward()

        grad_sign = images_for_attack.grad.detach().sign()
        adv_images = torch.clamp(images_for_attack.detach() + eps * grad_sign, 0.0, 1.0)

        # target model clean prediction
        with torch.no_grad():
            clean_outputs_target = target_model(normalize_batch(images))
            clean_preds_target = clean_outputs_target.argmax(dim=1)

            # target model adversarial prediction
            adv_outputs_target = target_model(normalize_batch(adv_images))
            adv_preds_target = adv_outputs_target.argmax(dim=1)

        target_clean_correct += (clean_preds_target == labels).sum().item()
        target_adv_correct += (adv_preds_target == labels).sum().item()

        attack_success_on_clean += (
            (clean_preds_target == labels) & (adv_preds_target != labels)
        ).sum().item()

        total += labels.size(0)

    clean_acc_ref = target_clean_correct / total
    adv_acc = target_adv_correct / total
    asr = attack_success_on_clean / target_clean_correct if target_clean_correct > 0 else 0.0

    graybox_fgsm_results.append({
        'attack': 'Gray-box FGSM',
        'surrogate': 'SimpleCNN',
        'target': 'ResNet18',
        'epsilon_255': e255,
        'clean_acc_reference': clean_acc_ref,
        'adv_acc_on_target': adv_acc,
        'attack_success_rate_on_target_clean_correct': asr
    })

graybox_fgsm_df = pd.DataFrame(graybox_fgsm_results)
graybox_fgsm_df

In [ ]:
graybox_fgsm_csv = os.path.join(save_dir, 'graybox_fgsm_summary.csv')
graybox_fgsm_df.to_csv(graybox_fgsm_csv, index=False)
print('Saved to:', graybox_fgsm_csv)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(graybox_fgsm_df['epsilon_255'], graybox_fgsm_df['adv_acc_on_target'] * 100, marker='o')
plt.axhline(target_clean_acc * 100, linestyle='--', label=f'Clean target acc = {target_clean_acc*100:.2f}%')

for _, row in graybox_fgsm_df.iterrows():
    plt.text(row['epsilon_255'], row['adv_acc_on_target'] * 100 + 1,
             f"{row['adv_acc_on_target']*100:.2f}%", ha='center')

plt.xlabel('Epsilon (/255)')
plt.ylabel('Accuracy on target model (%)')
plt.title('Gray-box FGSM Transfer Attack on ResNet-18')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def predict_with_target(pil_img):
    x = pixel_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = target_model(normalize_batch(x)).argmax(dim=1).item()
    return pred

In [ ]:
sample_idx = 0
eps_255 = 8
eps = eps_255 / 255.0

row = test_df.iloc[sample_idx]
img_path = os.path.join(extract_dir, row['Path'])
true_label = int(row['ClassId'])

img = Image.open(img_path).convert('RGB')
x = pixel_transform(img).unsqueeze(0).to(device)
y = torch.tensor([true_label], device=device)

x_attack = x.clone().detach().requires_grad_(True)
surrogate_model.zero_grad(set_to_none=True)
out = surrogate_model(normalize_batch(x_attack))
loss = criterion(out, y)
loss.backward()

adv_x = torch.clamp(x_attack.detach() + eps * x_attack.grad.sign(), 0.0, 1.0)

clean_img_np = x.squeeze(0).cpu().permute(1, 2, 0).numpy()
adv_img_np = adv_x.squeeze(0).cpu().permute(1, 2, 0).numpy()

clean_pred = predict_with_target(img)

adv_pil = transforms.ToPILImage()(adv_x.squeeze(0).cpu())
adv_pred = predict_with_target(adv_pil)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(clean_img_np)
axes[0].set_title(f'Clean\nTrue: {true_label} | Target Pred: {clean_pred}')
axes[0].axis('off')

axes[1].imshow(adv_img_np)
axes[1].set_title(f'Gray-box FGSM eps={eps_255}/255\nTrue: {true_label} | Target Pred: {adv_pred}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

2. PGD

In [ ]:
def graybox_pgd_attack(
    surrogate_model,
    images,
    labels,
    eps,
    alpha,
    num_steps,
    device='cuda',
    random_start=True
):
    """
    images: pixel-space images in [0,1]
    labels: ground-truth labels
    eps: max perturbation (e.g. 8/255)
    alpha: step size (e.g. 2/255)
    num_steps: number of PGD iterations
    """

    surrogate_model.eval()

    x_orig = images.clone().detach()

    if random_start:
        # random initialization within epsilon-ball
        x_adv = x_orig + torch.empty_like(x_orig).uniform_(-eps, eps)
        x_adv = torch.clamp(x_adv, 0.0, 1.0)
    else:
        x_adv = x_orig.clone().detach()

    for _ in range(num_steps):
        x_adv.requires_grad_(True)

        surrogate_model.zero_grad(set_to_none=True)
        outputs = surrogate_model(normalize_batch(x_adv))
        loss = criterion(outputs, labels)
        loss.backward()

        grad_sign = x_adv.grad.detach().sign()

        # gradient ascent step
        x_adv = x_adv.detach() + alpha * grad_sign

        # project back to epsilon-ball around original image
        delta = torch.clamp(x_adv - x_orig, min=-eps, max=eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()

    return x_adv

In [ ]:
eps_list_255 = [8]
eps_list = [e / 255.0 for e in eps_list_255]

# 보통 많이 쓰는 설정
alpha_255 = 2
alpha = alpha_255 / 255.0
num_steps = 5

In [ ]:
graybox_pgd_results = []

surrogate_model.eval()
target_model.eval()

for e255, eps in zip(eps_list_255, eps_list):
    total = 0
    target_clean_correct = 0
    target_adv_correct = 0
    attack_success_on_clean = 0

    for images, labels, paths in tqdm(attack_loader, desc=f'Gray-box PGD eps={e255}/255'):
        images = images.to(device, non_blocking=True)   # pixel-space [0,1]
        labels = labels.to(device, non_blocking=True)

        # PGD adversarial examples generated on surrogate model
        adv_images = graybox_pgd_attack(
            surrogate_model=surrogate_model,
            images=images,
            labels=labels,
            eps=eps,
            alpha=alpha,
            num_steps=num_steps,
            device=device,
            random_start=True
        )

        # evaluate on target model
        with torch.no_grad():
            clean_outputs_target = target_model(normalize_batch(images))
            clean_preds_target = clean_outputs_target.argmax(dim=1)

            adv_outputs_target = target_model(normalize_batch(adv_images))
            adv_preds_target = adv_outputs_target.argmax(dim=1)

        target_clean_correct += (clean_preds_target == labels).sum().item()
        target_adv_correct += (adv_preds_target == labels).sum().item()
        attack_success_on_clean += (
            (clean_preds_target == labels) & (adv_preds_target != labels)
        ).sum().item()
        total += labels.size(0)

    clean_acc_ref = target_clean_correct / total
    adv_acc = target_adv_correct / total
    asr = attack_success_on_clean / target_clean_correct if target_clean_correct > 0 else 0.0

    graybox_pgd_results.append({
        'attack': 'Gray-box PGD',
        'surrogate': 'SimpleCNN',
        'target': 'ResNet18',
        'epsilon_255': e255,
        'alpha_255': alpha_255,
        'num_steps': num_steps,
        'clean_acc_reference': clean_acc_ref,
        'adv_acc_on_target': adv_acc,
        'attack_success_rate_on_target_clean_correct': asr
    })

graybox_pgd_df = pd.DataFrame(graybox_pgd_results)
graybox_pgd_df

In [ ]:
graybox_pgd_csv = os.path.join(save_dir, 'graybox_pgd_summary.csv')
graybox_pgd_df.to_csv(graybox_pgd_csv, index=False)
print('Saved to:', graybox_pgd_csv)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(graybox_pgd_df['epsilon_255'], graybox_pgd_df['adv_acc_on_target'] * 100, marker='o', label='Gray-box PGD')
plt.axhline(target_clean_acc * 100, linestyle='--', label=f'Clean target acc = {target_clean_acc*100:.2f}%')

for _, row in graybox_pgd_df.iterrows():
    plt.text(row['epsilon_255'], row['adv_acc_on_target'] * 100 + 0.7,
             f"{row['adv_acc_on_target']*100:.2f}%", ha='center')

plt.xlabel('Epsilon (/255)')
plt.ylabel('Accuracy on target model (%)')
plt.title(f'Gray-box PGD Transfer Attack\n(alpha={alpha_255}/255, steps={num_steps})')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
#visualization
sample_idx = 0
eps_255 = 8
eps = eps_255 / 255.0

row = test_df.iloc[sample_idx]
img_path = os.path.join(extract_dir, row['Path'])
true_label = int(row['ClassId'])

img = Image.open(img_path).convert('RGB')
x = pixel_transform(img).unsqueeze(0).to(device)
y = torch.tensor([true_label], device=device)

adv_x = graybox_pgd_attack(
    surrogate_model=surrogate_model,
    images=x,
    labels=y,
    eps=eps,
    alpha=alpha,
    num_steps=num_steps,
    device=device,
    random_start=True
)

def predict_with_target_pil(pil_img):
    x_in = pixel_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = target_model(normalize_batch(x_in)).argmax(dim=1).item()
    return pred

clean_pred = predict_with_target_pil(img)
adv_pil = transforms.ToPILImage()(adv_x.squeeze(0).cpu())
adv_pred = predict_with_target_pil(adv_pil)

clean_np = x.squeeze(0).cpu().permute(1, 2, 0).numpy()
adv_np = adv_x.squeeze(0).cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(clean_np)
axes[0].set_title(f'Clean\nTrue: {true_label} | Target Pred: {clean_pred}')
axes[0].axis('off')

axes[1].imshow(adv_np)
axes[1].set_title(f'Gray-box PGD eps={eps_255}/255\nTrue: {true_label} | Target Pred: {adv_pred}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

3. Edge FGSM

In [ ]:
def sobel_edge_map(x):
    gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]

    sobel_x = torch.tensor(
        [[[-1, 0, 1],
          [-2, 0, 2],
          [-1, 0, 1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    sobel_y = torch.tensor(
        [[[-1, -2, -1],
          [ 0,  0,  0],
          [ 1,  2,  1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    gx = F.conv2d(gray, sobel_x, padding=1)
    gy = F.conv2d(gray, sobel_y, padding=1)

    magnitude = torch.sqrt(gx**2 + gy**2 + 1e-12)

    b = magnitude.size(0)
    mag_flat = magnitude.view(b, -1)
    mag_min = mag_flat.min(dim=1)[0].view(b, 1, 1, 1)
    mag_max = mag_flat.max(dim=1)[0].view(b, 1, 1, 1)

    edge_map = (magnitude - mag_min) / (mag_max - mag_min + 1e-8)
    return edge_map

In [ ]:
def build_edge_weight_map(x, beta=0.7):
    edge_map = sobel_edge_map(x)
    weight_map = (1.0 - beta) + beta * edge_map
    weight_map = weight_map.repeat(1, 3, 1, 1)
    return edge_map, weight_map

In [ ]:
def graybox_edge_fgsm_attack(
    surrogate_model,
    images,
    labels,
    eps,
    beta=0.7
):
    surrogate_model.eval()

    x = images.clone().detach().requires_grad_(True)

    surrogate_model.zero_grad(set_to_none=True)
    outputs = surrogate_model(normalize_batch(x))
    loss = criterion(outputs, labels)
    loss.backward()

    grad_sign = x.grad.detach().sign()

    with torch.no_grad():
        _, weight_map = build_edge_weight_map(x.detach(), beta=beta)

        adv_images = torch.clamp(
            x.detach() + eps * weight_map * grad_sign,
            0.0,
            1.0
        )

    return adv_images

In [ ]:
edge_eps_list_255 = [8]
edge_eps_list = [8 / 255.0]
edge_beta = 0.7

In [ ]:
# =========================
# Gray-box Edge-FGSM
# =========================

# 1) Sobel edge map
def sobel_edge_map(x):
    # x: [B, 3, H, W], pixel-space [0,1]
    gray = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]

    sobel_x = torch.tensor(
        [[[-1, 0, 1],
          [-2, 0, 2],
          [-1, 0, 1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    sobel_y = torch.tensor(
        [[[-1, -2, -1],
          [ 0,  0,  0],
          [ 1,  2,  1]]],
        dtype=x.dtype,
        device=x.device
    ).unsqueeze(0)

    gx = F.conv2d(gray, sobel_x, padding=1)
    gy = F.conv2d(gray, sobel_y, padding=1)

    magnitude = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

    # normalize each image to [0,1]
    b = magnitude.size(0)
    mag_flat = magnitude.view(b, -1)
    mag_min = mag_flat.min(dim=1)[0].view(b, 1, 1, 1)
    mag_max = mag_flat.max(dim=1)[0].view(b, 1, 1, 1)

    edge_map = (magnitude - mag_min) / (mag_max - mag_min + 1e-8)
    return edge_map


# 2) Edge weight map
def build_edge_weight_map(x, beta=0.7):
    edge_map = sobel_edge_map(x)           # [B,1,H,W]
    weight_map = (1.0 - beta) + beta * edge_map
    weight_map = weight_map.repeat(1, 3, 1, 1)  # RGB 채널로 확장
    return edge_map, weight_map


# 3) Gray-box Edge-FGSM attack
def graybox_edge_fgsm_attack(
    surrogate_model,
    images,
    labels,
    eps,
    beta=0.7
):
    """
    surrogate_model: attack gradient를 계산할 surrogate model
    images: pixel-space images in [0,1]
    labels: ground-truth labels
    eps: perturbation budget (e.g. 8/255)
    beta: edge emphasis strength
    """
    surrogate_model.eval()

    x = images.clone().detach().requires_grad_(True)

    surrogate_model.zero_grad(set_to_none=True)
    outputs = surrogate_model(normalize_batch(x))
    loss = criterion(outputs, labels)
    loss.backward()

    grad_sign = x.grad.detach().sign()

    with torch.no_grad():
        _, weight_map = build_edge_weight_map(x.detach(), beta=beta)
        adv_images = torch.clamp(
            x.detach() + eps * weight_map * grad_sign,
            0.0,
            1.0
        )

    return adv_images


# 4) 먼저 epsilon 하나만 테스트
edge_eps_list_255 = [8]
edge_eps_list = [e / 255.0 for e in edge_eps_list_255]
edge_beta = 0.7


# 5) Evaluation loop
graybox_edge_results = []

surrogate_model.eval()
target_model.eval()

for e255, eps in zip(edge_eps_list_255, edge_eps_list):
    total = 0
    target_clean_correct = 0
    target_adv_correct = 0
    attack_success_on_clean = 0

    for images, labels, paths in tqdm(attack_loader, desc=f'Gray-box Edge-FGSM eps={e255}/255'):
        images = images.to(device, non_blocking=True)   # pixel-space [0,1]
        labels = labels.to(device, non_blocking=True)

        # gray-box edge attack: gradient는 surrogate model에서 계산
        adv_images = graybox_edge_fgsm_attack(
            surrogate_model=surrogate_model,
            images=images,
            labels=labels,
            eps=eps,
            beta=edge_beta
        )

        # target model evaluation
        with torch.no_grad():
            clean_outputs_target = target_model(normalize_batch(images))
            clean_preds_target = clean_outputs_target.argmax(dim=1)

            adv_outputs_target = target_model(normalize_batch(adv_images))
            adv_preds_target = adv_outputs_target.argmax(dim=1)

        target_clean_correct += (clean_preds_target == labels).sum().item()
        target_adv_correct += (adv_preds_target == labels).sum().item()

        attack_success_on_clean += (
            (clean_preds_target == labels) & (adv_preds_target != labels)
        ).sum().item()

        total += labels.size(0)

    clean_acc_ref = target_clean_correct / total
    adv_acc = target_adv_correct / total
    asr = attack_success_on_clean / target_clean_correct if target_clean_correct > 0 else 0.0

    graybox_edge_results.append({
        'attack': 'Gray-box Edge-FGSM',
        'surrogate': 'SimpleCNN',
        'target': 'ResNet18',
        'epsilon_255': e255,
        'beta': edge_beta,
        'clean_acc_reference': clean_acc_ref,
        'adv_acc_on_target': adv_acc,
        'attack_success_rate_on_target_clean_correct': asr
    })

graybox_edge_df = pd.DataFrame(graybox_edge_results)
graybox_edge_df

In [ ]:
#Save result
graybox_edge_csv = os.path.join(save_dir, 'graybox_edge_fgsm_summary.csv')
graybox_edge_df.to_csv(graybox_edge_csv, index=False)
print('Saved to:', graybox_edge_csv)
print(graybox_edge_df)

In [ ]:
# 7) Plot
plt.figure(figsize=(6, 4))
plt.plot(graybox_edge_df['epsilon_255'], graybox_edge_df['adv_acc_on_target'] * 100, marker='o', label='Gray-box Edge-FGSM')
plt.axhline(target_clean_acc * 100, linestyle='--', label=f'Clean target acc = {target_clean_acc*100:.2f}%')

for _, row in graybox_edge_df.iterrows():
    plt.text(row['epsilon_255'], row['adv_acc_on_target'] * 100 + 0.7,
             f"{row['adv_acc_on_target']*100:.2f}%", ha='center')

plt.xlabel('Epsilon (/255)')
plt.ylabel('Accuracy on target model (%)')
plt.title(f'Gray-box Edge-FGSM Transfer Attack\n(beta={edge_beta})')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 8) Sample visualization
sample_idx = 0
eps_255 = 8
eps = eps_255 / 255.0

row = test_df.iloc[sample_idx]
img_path = os.path.join(extract_dir, row['Path'])
true_label = int(row['ClassId'])

img = Image.open(img_path).convert('RGB')
x = pixel_transform(img).unsqueeze(0).to(device)
y = torch.tensor([true_label], device=device)

adv_x = graybox_edge_fgsm_attack(
    surrogate_model=surrogate_model,
    images=x,
    labels=y,
    eps=eps,
    beta=edge_beta
)

def predict_target_from_tensor(x_tensor):
    with torch.no_grad():
        pred = target_model(normalize_batch(x_tensor)).argmax(dim=1).item()
    return pred

clean_pred = predict_target_from_tensor(x)
adv_pred = predict_target_from_tensor(adv_x)

clean_np = x.squeeze(0).cpu().permute(1, 2, 0).numpy()
adv_np = adv_x.squeeze(0).cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(clean_np)
axes[0].set_title(f'Clean\nTrue: {true_label} | Target Pred: {clean_pred}')
axes[0].axis('off')

axes[1].imshow(adv_np)
axes[1].set_title(f'Gray-box Edge-FGSM eps={eps_255}/255\nTrue: {true_label} | Target Pred: {adv_pred}')
axes[1].axis('off')

plt.tight_layout()
plt.show()